# VulnLocAI — Inference & Evaluation

Evaluates the fine-tuned VulnLocAI model on the **SVD-Benchmark** (5,053 samples from
20 real-world Java projects).

**Metrics:**
| Metric | Definition |
|--------|------------|
| Precision | TP / (TP + FP) |
| Recall | TP / (TP + FN) |
| F1-Score | 2·P·R / (P+R) |
| Accuracy | (TP+TN) / total |
| Top-1 Accuracy | predicted line contained within GT vulnerable line |
| Exact Match (EM) | predicted line == GT line (exact string, after normalisation) |


In [ ]:
# Install dependencies (run once):
# !pip install unsloth transformers datasets scikit-learn tqdm --quiet
from unsloth import FastLanguageModel
import torch, pandas as pd, sys, json
from tqdm import tqdm
sys.path.insert(0, ".")
from metrics_utils import parse_model_output, compute_all_metrics, top1_accuracy, exact_match


## 1. Load the Fine-Tuned VulnLocAI Model

Loads the LoRA adapters saved by `VulnLocAI.ipynb` from `lora_model/`.

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="lora_model",       # saved LoRA checkpoint from VulnLocAI.ipynb
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)   # enable fast inference mode
model.eval()
print("VulnLocAI model loaded.")


## 2. Alpaca Inference Prompt

Same template as training (Table 2 of the paper). The `### Response:` section is left empty so the model generates the prediction. The instruction does **not** pre-supply the CWE ID.

In [ ]:
INSTRUCTION = "Analyze the following Java code snippet and identify any security vulnerability"

ALPACA_INFERENCE_PROMPT = """### Instruction:
{instruction}

### Input:
{code_snippet}

### Response:
"""

def build_prompt(code_snippet: str) -> str:
    return ALPACA_INFERENCE_PROMPT.format(
        instruction=INSTRUCTION,
        code_snippet=code_snippet,
    )


## 3. Inference Function

Greedy decoding (`do_sample=False`) for reproducibility. `max_new_tokens=256` is sufficient for the structured output (CWE-ID + one line + short description).

In [ ]:
def predict(code_snippet: str, max_new_tokens: int = 256) -> str:
    prompt = build_prompt(code_snippet)
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=max_seq_length,
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                    # greedy — reproducible
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


## 4. Load SVD-Benchmark

5,053 samples from 20 real-world Java projects. Entirely disjoint from the 907 training projects.

In [ ]:
benchmark_df = pd.read_csv("Evaluation-Benchmark/SVD-Benchmark.csv")
print(f"Loaded {len(benchmark_df)} samples")
print(benchmark_df["Status"].value_counts().to_string())
print("Projects:", benchmark_df["Project Name"].nunique())


## 5. Run Inference on All Samples

In [ ]:
raw_outputs, parsed_predictions = [], []

for _, row in tqdm(benchmark_df.iterrows(), total=len(benchmark_df), desc="VulnLocAI inference"):
    raw = predict(str(row["Code Snippet"]))
    parsed = parse_model_output(raw)
    raw_outputs.append(raw)
    parsed_predictions.append(parsed)

benchmark_df["raw_prediction"]  = raw_outputs
benchmark_df["predicted_label"] = [p["predicted_label"] for p in parsed_predictions]
benchmark_df["predicted_cwe"]   = [p["predicted_cwe"]   for p in parsed_predictions]
benchmark_df["predicted_line"]  = [p["predicted_line"]  for p in parsed_predictions]
print("Inference complete.")


## 6. Compute All Metrics (Table 3 of the paper)

In [ ]:
results = compute_all_metrics(parsed_predictions, benchmark_df, model_name="VulnLocAI")


## 7. Per-Project Recall (Figure 5 of the paper)

In [ ]:
project_results = []
for project, group in benchmark_df[benchmark_df["Status"] == "vulnerable"].groupby("Project Name"):
    total    = len(group)
    detected = (group["predicted_label"] == "vulnerable").sum()
    project_results.append({
        "Project":  project,
        "Total":    total,
        "Detected": int(detected),
        "Recall%":  round(detected / total * 100, 1) if total else 0.0,
    })
proj_df = pd.DataFrame(project_results).sort_values("Recall%", ascending=False)
print(proj_df.to_string(index=False))


## 8. Per-CWE Analysis (Figure 4 / Table 4)

In [ ]:
cwe_results = []
for cwe, group in benchmark_df[benchmark_df["Status"] == "vulnerable"].groupby("CWE ID"):
    total    = len(group)
    detected = (group["predicted_label"] == "vulnerable").sum()
    em_hits  = sum(
        1 for _, r in group.iterrows()
        if r["predicted_label"] == "vulnerable"
        and str(r.get("predicted_line","")).strip().lower()
           == str(r["Exact Vulnerable Line"]).strip().lower()
    )
    cwe_results.append({"CWE": cwe, "Total": total,
                        "Recall%": round(detected/total*100,1),
                        "EM%":     round(em_hits/total*100,1)})
cwe_df = pd.DataFrame(cwe_results).sort_values("Total", ascending=False)
print(cwe_df.to_string(index=False))


## 9. Save Results

In [ ]:
benchmark_df.to_csv("Results/VulnLocAI_predictions.csv", index=False)
proj_df.to_csv("Results/VulnLocAI_by_project.csv", index=False)
cwe_df.to_csv("Results/VulnLocAI_by_cwe.csv", index=False)
with open("Results/VulnLocAI_summary_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved to Results/")
